In [1]:
import transformers
from transformers import PreTrainedModel, PreTrainedTokenizer

In [2]:
import torch
import numpy
import random
import gc


def set_seed(seed: int) -> None:
    """
    Set the seed for reproducibility.

    Args:
        seed (int): Seed to set.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    numpy.random.seed(seed)
    random.seed(seed)


def get_device() -> torch.device:
    """
    Get the device to use for computations.
    """
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def clear_memory() -> None:
    """
    Frees unused memory by calling the garbage collector and clearing the CUDA cache.
    This helps prevent out-of-memory errors in GPU-limited environments.
    """
    gc.collect()
    torch.cuda.empty_cache()


def extract_device(module: torch.nn.Module) -> torch.device:
    """
    Extract the device from a module.
    """
    return next(iter(module.parameters())).device


def sample_lp_ball(length: int, norm: float = 2.0, device: torch.device | None = None) -> torch.Tensor:
    """
    Implementation of the method described in:
    [https://stats.stackexchange.com/questions/352668/generate-uniform-noise-from-a-p-norm-ball-x-p-leq-r]

    Sample a random vector from the Lp ball of radius 1.

    Args:
        length (int): Length of the vector.
        norm (int): Norm of the ball.
        device (torch.device, optional): Device to move the tensor to.

    Returns:
        torch.Tensor: A random vector sampled from the Lp ball.
    """
    if device is None:
        device = torch.device("cpu")
    vec = (-torch.log(torch.rand(length, device=device))) ** (1 / norm)
    sgn = 2 * torch.randint(0, 2, (length,), dtype=torch.float32, device=device) - 1
    vec = sgn * vec
    vec = vec / (torch.norm(vec, p=norm) + torch.finfo(vec.dtype).eps)
    rad = torch.exp(torch.log(torch.rand(1, device=device)) / length)
    return rad * vec

In [ ]:
class EmbedAttack:
    def __init__(
        self,
        model: PreTrainedModel,
        tokenizer: PreTrainedTokenizer,
        num_tokens: int = 10,
    ):
        # model
        self.model = model.eval()
        self.model.requires_grad_(False)
        self.device = extract_device(model)

        # tokenizer
        self.tokenizer = tokenizer
        self.adv_token = "[ADV]"
        if self.adv_token not in tokenizer.get_vocab():
            tokenizer.add_special_tokens({"additional_special_tokens": [self.adv_token]})
            self.model.resize_token_embeddings(len(tokenizer))

        # attack params
        self.num_tokens = num_tokens

    def tokenize_input_target(self, input_texts: list[str], target_texts: list[str]):
        """
        Tokenize the input and target texts.
        This function pads the input and target texts such that the adversarial tokens are aligned across all samples.

        Args:
            input_texts (list[str]): List of input texts.
            target_texts (list[str]): List of target texts.

        Returns:
            dict: Dictionary containing the tokenized input and target texts, with the following keys:
                - input_ids: Token IDs of the input texts.
                - attention_mask: Attention mask for the input texts.
                - adv_mask: Mask for the adversarial tokens.
                - target_mask: Mask for the target texts.
        """

        # we pad from input and target side, such that the adv tokens are aligned across all samples
        # this allows us to use KV-cache efficiently for all samples, and ease of access to adv embedding
        
        # example (P - padding, I - input, A - adv, T - target):
        # [P][P][P][I][I] [A][A][A] [T][T][P][P]
        # [P][I][I][I][I] [A][A][A] [T][T][T][T]
        # [I][I][I][I][I] [A][A][A] [T][P][P][P]
        
        # tested on:
        # - meta-llama/Llama-3.2-1B-Instruct
        # - Qwen/Qwen3-0.6B
        # - samwit/koala-7b - target should begin with <think> token

        input_messeges = []
        for inp_txt in input_texts:
            msg = [{"role": "user", "content": inp_txt + (self.adv_token * self.num_tokens)}]
            input_messeges.append(msg)

        tokenizer.padding_side = "left"
        input_tokens = self.tokenizer.apply_chat_template(
            input_messeges,
            add_generation_prompt=True,
            padding=True,
            padding_side="left",
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)

        tokenizer.padding_side = "right"
        target_tokens = self.tokenizer(
            target_texts,
            padding=True,
            padding_side="right",
            return_tensors="pt",
            return_attention_mask=True,
        ).to(self.device)

        # check if BOS was added to target, if yes remove it
        if self.tokenizer.bos_token and target_tokens["input_ids"][0][0] == self.tokenizer.bos_token_id:
            target_tokens["input_ids"] = target_tokens["input_ids"][:, 1:]
            target_tokens["attention_mask"] = target_tokens["attention_mask"][:, 1:]

        # combine input and target tokens
        token_ids = torch.cat([input_tokens["input_ids"], target_tokens["input_ids"]], dim=1)
        attn_mask = torch.cat([input_tokens["attention_mask"], target_tokens["attention_mask"]], dim=1)

        # create adv token mask
        adv_token_id = self.tokenizer.convert_tokens_to_ids(self.adv_token)
        adv_mask = token_ids == adv_token_id

        # create target mask
        target_mask = torch.zeros_like(token_ids, dtype=torch.bool, device=self.device)
        target_mask[:, -target_tokens["input_ids"].shape[1] :] = True
        target_mask = target_mask & ~attn_mask

        return {
            "input_ids": token_ids,
            "attention_mask": attn_mask,
            "adv_mask": adv_mask,
            "target_mask": target_mask,
        }

    def tokenize_input(self, input_texts: list[str]):
        """
        Tokenize the input texts.

        Args:
            input_texts (list[str]): List of input texts.

        Returns:
            dict: Dictionary containing the tokenized input texts with the following keys:
                - input_ids: Token IDs of the input texts.
                - attention_mask: Attention mask for the input texts.
                - adv_mask: Mask for the adversarial tokens.

        """
        input_messeges = []
        for inp_txt in input_texts:
            msg = [{"role": "user", "content": inp_txt + (self.adv_token * self.num_tokens)}]
            input_messeges.append(msg)

        tokenizer.padding_side = "left"
        input_tokens = self.tokenizer.apply_chat_template(
            input_messeges,
            add_generation_prompt=True,
            padding=True,
            padding_side="left",
            return_dict=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(self.device)

        # create adv token mask
        adv_token_id = self.tokenizer.convert_tokens_to_ids(self.adv_token)
        adv_mask = token_ids == adv_token_id

        return {
            "input_ids": input_tokens["input_ids"],
            "attention_mask": input_tokens["attention_mask"],
            "adv_mask": adv_mask,
        }
        
    def embed(self, token_ids: torch.Tensor) -> torch.Tensor:
        embedding_layer = model.get_input_embeddings()
        return embedding_layer(token_ids)

    def fit(self, input_texts: list[str], target_texts: list[str]):
        """
        Fit the attack model to the input and target texts.

        Args:
            input_texts (list[str]): List of input texts.
            target_texts (list[str]): List of target texts.
        """
        
        # TODO: FIX BUG WITH GRADIENTS, WTF
        # TODO: actually save KV-cache for the input texts
        
        token_dict = self.tokenize_input_target(input_texts, target_texts)
        inputs_embeds = self.embed(token_dict["input_ids"])
        
        adv_embed = torch.randn((len(input_texts), self.num_tokens, inputs_embeds.shape[-1]), device=self.device)
        
        optim = torch.optim.Adam([adv_embed], lr=1e-4)
        
        for i in range(10):
            optim.zero_grad()
            
            inputs_embeds = inputs_embeds.masked_scatter(mask=token_dict["adv_mask"].unsqueeze(-1), source=adv_embed)
            
            result = self.model(
                input_ids=None,
                inputs_embeds=inputs_embeds,
                # attention_mask=token_dict["attention_mask"], # maybe we dont need it?
                output_hidden_states=True, # do we need it? 
            )
            
            logits = result.logits
            
            # shift logits and labels
            # probably there is a better way to do this
            input_logits = logits.roll(1, dims=1).contiguous()
            input_ids = token_dict["input_ids"].contiguous()
            
            pred_logits = input_logits[token_dict["target_mask"]]
            target_ids = input_ids[token_dict["target_mask"]]
            
            loss = torch.nn.functional.cross_entropy(
                pred_logits.view(-1, pred_logits.size(-1)), target_ids.view(-1)
            )
            
            loss_float = loss.item()
            print(f"Iter: {i} Loss: {loss_float}")
            
            grad = torch.autograd.grad(loss, adv_embed, retain_graph=False, create_graph=False)
            optim.step()
        
        

In [4]:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from torch import optim

# # model_name = "Qwen/Qwen3-0.6B"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
# # model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# # model_name = "samwit/koala-7b"


# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name)

# if not tokenizer.pad_token:
#     tokenizer.pad_token = tokenizer.eos_token

# attk = EmbedAttack(
#     model=model,
#     tokenizer=tokenizer,
#     num_tokens=0,
# )

# inputs = ["Translate to French: Hello!", "How are you?", "What is the meaning of life?"]
# labels = ["Bonjour !", "Im good, thanks!", "45 actually"]


# usr_msg = [[{"role": "user", "content": inp}] for inp in inputs]

# tokenizer.padding_side = "left"
# user_side = tokenizer.apply_chat_template(
#     usr_msg,
#     add_generation_prompt=True,
#     padding=True,
#     padding_side="left",
#     return_dict=True,
#     return_tensors="pt",
#     return_attention_mask=True,
#     enable_thinking=False,
# )

# tokenizer.padding_side = "right"
# assistant_side = tokenizer.__call__(
#     # [[lbl] for lbl in labels],
#     labels,
#     padding=True,
#     padding_side="right",
#     return_tensors="pt",
#     return_attention_mask=True,
# )

# if tokenizer.bos_token is not None and assistant_side["input_ids"][0][0] == tokenizer.bos_token_id:
#     combined_tensor = torch.cat((user_side["input_ids"], assistant_side["input_ids"][:, 1:]), dim=1)
# else:
#     combined_tensor = torch.cat((user_side["input_ids"], assistant_side["input_ids"]), dim=1)

# combined_messege = tokenizer.batch_decode(combined_tensor, skip_special_tokens=False)

# both_msg = [[{"role": "user", "content": inp}, {"role": "assistant", "content": lbl}] for inp, lbl in zip(inputs, labels)]

# tokenizer.padding_side = "left"
# at_once = tokenizer.apply_chat_template(
#     both_msg,
#     # add_generation_prompt=False,
#     padding=True,
#     padding_side="left",
#     continue_final_message=True,
#     enable_thinking=False,
#     return_attention_mask=True,
#     return_tensors="pt",
# )

# at_once = tokenizer.batch_decode(at_once, skip_special_tokens=False)

# at_once_2 = tokenizer.batch_decode(attk.tokenize_input_target(inputs, labels)["input_ids"], skip_special_tokens=False)

# for msg_combined, msg_at_once, msg_at_once_2 in zip(combined_messege, at_once, at_once_2):
#     print("Combined:", [msg_combined])
#     print("Comb2   :", [msg_at_once_2])
#     print("At Once :", [msg_at_once])
#     print(msg_combined == msg_at_once)
#     print(msg_combined == msg_at_once_2)
#     print()

# # for msg in at_once:
# #     print([msg])
# # print("Combined At Once:\n", at_once)

# # attk.tokenize_input_target(inputs, labels)

In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from torch import optim

# model_name = "Qwen/Qwen3-0.6B"
model_name = "meta-llama/Llama-3.2-1B-Instruct"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
# model_name = "samwit/koala-7b"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to("cuda")

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

attk = EmbedAttack(
    model=model,
    tokenizer=tokenizer,
    num_tokens=5,
)

inputs = ["Translate to French: Hello!", "How are you?", "What is the meaning of life?"]
labels = ["Bonjour !", "Im good, thanks!", "45 actually"]

attk.fit(inputs, labels)

Iter: 0 Loss: 6.956320285797119
Iter: 1 Loss: 6.956320285797119


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.